# RNN Gerçek Dünya Örneği: Hisse Senedi Fiyat Tahmini

## Senaryo

Bir **fintech şirketinde** çalıştığını düşün.  
Görevin: Hisse senedi fiyatlarının gelecek trendini tahmin eden bir model geliştirmek.

**Sorun:** Hisse fiyatı tamamen bağımsız değil — **dünün, geçen haftanın, geçen ayın** fiyatı bugünkü fiyatı etkiler.  
**Çözüm:** RNN/LSTM — geçmişteki desenleri öğrenerek gelecekteki hareketi tahmin et.

## Neden RNN/LSTM?

- **FNN**: Her günü bağımsız işler. "Dün ne oldu?" sorusunu soramaz.
- **CNN**: Görüntüdeki uzamsal desenleri yakalar ama zaman bağlamını bilmez.
- **RNN/LSTM**: Zaman içindeki sıralı bağımlılıkları öğrenir. "Son 60 günde ne oldu?" sorusunu yanıtlar.

## SimpleRNN vs LSTM vs GRU

| Model     | Hafıza | Hız   | Uzun bağımlılık |
| --------- | ------ | ----- | --------------- |
| SimpleRNN | Kısa   | Hızlı | Zayıf           |
| **LSTM**  | Uzun   | Orta  | Güçlü           |
| GRU       | Orta   | Orta  | İyi             |

Bu notebook'ta hem **SimpleRNN** hem **LSTM** kullanarak karşılaştırma yapacağız.

> **Önemli Not:** Bu model eğitim amaçlıdır. Gerçek hisse yatırım kararları için kullanmayın!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")

# --- nn3d: agi tarayicida canli 3D izlemek icin ---------------------------
import sys, pathlib
if not any(pathlib.Path(p, "nn3d").is_dir() for p in sys.path):
    sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "src"))
import nn3d


## Adım 1: Gerçekçi Hisse Senedi Verisi Oluştur

Gerçek hisse verisi çekmek için `yfinance` kütüphanesi gerekir.  
Bu notebook'ta her ortamda çalışsın diye **gerçek hisse davranışını simüle eden** sentetik veri oluşturuyoruz:

- **Trend**: Genel yükseliş eğilimi (uzun vadeli büyüme)
- **Mevsimsellik**: Yılın belirli dönemlerinde tekrar eden örüntüler
- **Gürültü**: Günlük dalgalanmalar (piyasa gürültüsü)
- **Volatilite**: Zaman zaman artan/azalan dalgalanma

Bu bileşenlerin kombinasyonu gerçek hisse grafiklerine çok benzer bir yapı üretir.


In [ ]:
# Gerçekçi hisse senedi fiyat verisi üret
# 5 yıl = ~1260 iş günü
n_days = 1260
dates = pd.date_range(start='2019-01-01', periods=n_days, freq='B')  # B = iş günü

# Bileşen 1: Uzun vadeli trend (yavaş yükseliş)
trend = np.linspace(100, 180, n_days)

# Bileşen 2: Mevsimsel döngü (yılda bir tam tur)
seasonality = 15 * np.sin(2 * np.pi * np.arange(n_days) / 252)  # 252 iş günü/yıl

# Bileşen 3: Kümülatif rastgele yürüyüş (Brownian motion — gerçek hisse modeli)
daily_returns = np.random.normal(0.0003, 0.015, n_days)  # Ortalama %0.03 günlük getiri
random_walk = np.cumsum(daily_returns) * 30

# Bileşen 4: Volatilite dalgalanması (kriz dönemleri)
volatility = np.ones(n_days)
volatility[400:450] = 3.0   # 2020 başı kriz
volatility[750:780] = 2.0   # İkinci volatilite artışı
crisis_noise = np.random.normal(0, 1, n_days) * volatility * 3

# Fiyat = Trend + Mevsimsellik + Rastgele Yürüyüş + Kriz Gürültüsü
price = trend + seasonality + random_walk + crisis_noise
price = np.clip(price, 50, 300)   # Fiyatı mantıklı sınırlar içinde tut

# DataFrame oluştur
df = pd.DataFrame({
    'Date': dates,
    'Close': price,
    'Volume': np.random.randint(1_000_000, 5_000_000, n_days)   # Hacim verisi
})
df.set_index('Date', inplace=True)

print(f"Veri aralığı: {df.index[0].date()} → {df.index[-1].date()}")
print(f"Toplam iş günü: {len(df)}")
print(f"\nFiyat istatistikleri:")
print(df['Close'].describe().round(2))

In [ ]:
# Ham fiyat verisini görselleştir
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Fiyat grafiği
axes[0].plot(df.index, df['Close'], color='#2196F3', linewidth=1)
axes[0].fill_between(df.index, df['Close'], alpha=0.1, color='#2196F3')
axes[0].set_title('Simüle Edilmiş Hisse Senedi Fiyatı (5 Yıl)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Fiyat ($)')
axes[0].grid(True, alpha=0.3)

# 30 günlük hareketli ortalama ekle
ma30 = df['Close'].rolling(window=30).mean()
axes[0].plot(df.index, ma30, color='#FF5722', linewidth=2, 
             label='30 Günlük Hareketli Ortalama', linestyle='--')
axes[0].legend()

# Hacim grafiği
axes[1].bar(df.index, df['Volume'] / 1e6, color='#9C27B0', alpha=0.6, width=1)
axes[1].set_ylabel('Hacim (Milyon)')
axes[1].set_xlabel('Tarih')
axes[1].grid(True, alpha=0.3)
axes[1].set_title('İşlem Hacmi')

plt.tight_layout()
plt.show()

## Adım 2: Veri Ön İşleme

### MinMaxScaler — Neden StandardScaler Değil?

Hisse fiyatları için **MinMaxScaler** tercih edilir:

- Tüm değerleri `[0, 1]` aralığına çeker
- LSTM hücreleri sigmoid/tanh içerir → bu aralıkta daha iyi çalışır
- StandardScaler negatif değerler üretebilir (fiyat için anlamsız)

### Zaman Serisi Train/Test Bölmesi

**ÖNEMLI FARK:** Normal sınıflandırmada rastgele böleriz. Zaman serisinde ASLA rastgele bölme yapma!  
Nedeni: Gelecek verisi eğitim setine karışırsa model "geleceğe bakıyor" olur (**data leakage**).

```
Doğru:  [===Eğitim (80%)===][==Test (20%)==]  → Kronolojik sıra korunur
Yanlış: Rastgele karıştırma → Gelecek eğitime karışır
```


In [ ]:
# Sadece kapanış fiyatını al
prices = df['Close'].values.reshape(-1, 1)

# MinMaxScaler: [0, 1] aralığına normalize et
scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(prices)

# KRONOLOJİK bölme — ilk %80 eğitim, son %20 test
split_idx = int(len(prices_scaled) * 0.80)
train_data = prices_scaled[:split_idx]
test_data  = prices_scaled[split_idx:]

print(f"Eğitim verisi: {len(train_data)} gün ({df.index[0].date()} → {df.index[split_idx].date()})")
print(f"Test verisi:   {len(test_data)} gün ({df.index[split_idx].date()} → {df.index[-1].date()})")
print(f"\nNormalize edilmiş aralık: {prices_scaled.min():.3f} - {prices_scaled.max():.3f}")

In [ ]:
# Dizi oluştur: son 60 günü gör, 61. günü tahmin et
SEQ_LENGTH = 60   # 3 aylık işlem günü

def create_sequences(data, seq_len):
    """Zaman serisini (X, y) pencerelerine böler."""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])          # Pencere: 60 gün
        y.append(data[i + seq_len])               # Hedef: sonraki gün
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_test, y_test   = create_sequences(test_data, SEQ_LENGTH)

print(f"X_train şekli: {X_train.shape}  → (örnek, zaman_adımı, özellik)")
print(f"y_train şekli: {y_train.shape}")
print(f"X_test şekli:  {X_test.shape}")
print(f"\nModel: Son {SEQ_LENGTH} günün fiyatına bakarak yarınki fiyatı tahmin et")

## Adım 3: İki Model — SimpleRNN vs LSTM

### LSTM (Long Short-Term Memory) Nedir?

SimpleRNN'nin büyük problemi: **vanishing gradient** (kaybolan gradyan).  
100 adım önceki bilgiyi hatırlayamaz. 10-15 adım ötesi bulanıklaşır.

LSTM bunu **3 kapı** mekanizmasıyla çözer:

```
Forget Gate  → "Bu eski bilgiyi ne kadarını unut?"
Input Gate   → "Bu yeni bilginin ne kadarını ezberle?"
Output Gate  → "Bu hafızadan ne kadarını çıkışa ver?"
```

Bu mekanizma sayesinde LSTM 100-200 adım öncesini hatırlayabilir.


In [ ]:
# Model 1: SimpleRNN
model_rnn = models.Sequential([
    layers.SimpleRNN(64, input_shape=(SEQ_LENGTH, 1), return_sequences=True),
    # return_sequences=True: her zaman adımında çıkış ver (bir sonraki RNN için)
    layers.SimpleRNN(32),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)   # Tek değer tahmini — yarınki fiyat
], name='SimpleRNN_Model')

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])
print("SimpleRNN:")
model_rnn.summary()
print("\n" + "="*60)

In [ ]:
# Model 2: LSTM (Long Short-Term Memory)
model_lstm = models.Sequential([
    # İlk LSTM katmanı — dizi çıkışı ver (stacked LSTM için)
    layers.LSTM(64, input_shape=(SEQ_LENGTH, 1), return_sequences=True),
    layers.Dropout(0.2),
    
    # İkinci LSTM katmanı — son adımın çıkışını ver
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
], name='LSTM_Model')

model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
print("LSTM:")
model_lstm.summary()

## Canlı 3D görselleştirmeRNN'lerde dikkat edilecek iki şey var:- **Mor halkalar** — RNN/LSTM katmanlarının kendine dönen (recurrent) bağlantısı.  Ağın "hafızası" bu döngüde.- **`return_sequences=True`** olan katman `(60, 64)` çıktı verir; nn3d bunun  **son eksenini** (64 birim) çizer, 3840'ı değil. Zaman ekseni kartta metin  olarak görünür.LSTM'in kapı ağırlıkları (input/forget/cell/output) görselleştirme içinortalanır — amaç sayısal doğruluk değil, bağlantının işareti ve şiddeti.

In [ ]:
# EarlyStopping ile her iki modeli eğit
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True
)

# nn3d: RNN katmanlarının kendine dönen bağlantısı mor halkalarla çizilir.
# İki modeli sırayla izliyoruz — LSTM eğitimi başlayınca SimpleRNN'in
# sekmesi kapanır ve aynı adreste LSTM görünür (aynı anda tek sunucu).
print("SimpleRNN eğitimi...")
izleyici_rnn = nn3d.Monitor(X_train[:1], every=10)
history_rnn = model_rnn.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop, izleyici_rnn],
    verbose=0   # Çıktıyı temiz tut
)
rnn_epochs = len(history_rnn.history['loss'])
print(f"SimpleRNN: {rnn_epochs} epoch'ta tamamlandı")

print("\nLSTM eğitimi...")
izleyici_lstm = nn3d.Monitor(X_train[:1], every=10)
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop, izleyici_lstm],
    verbose=0
)
lstm_epochs = len(history_lstm.history['loss'])
print(f"LSTM: {lstm_epochs} epoch'ta tamamlandı")

In [ ]:
# Eğitim kayıp karşılaştırması
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, history, name, color in zip(
    axes,
    [history_rnn, history_lstm],
    ['SimpleRNN', 'LSTM'],
    ['#FF5722', '#4CAF50']
):
    ax.plot(history.history['loss'], label='Eğitim MSE', color=color, linewidth=2)
    ax.plot(history.history['val_loss'], label='Doğrulama MSE', 
            color=color, linestyle='--', alpha=0.7)
    ax.set_title(f'{name} — Eğitim Kaybı')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('SimpleRNN vs LSTM Eğitim Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Adım 4: Tahminler ve Değerlendirme

**Önemli:** Model normalize edilmiş değerler üzerinde eğitildi.  
Gerçek fiyatlarla karşılaştırmak için `inverse_transform` ile geri dönüştürmemiz gerekiyor.


In [ ]:
# Tahminleri gerçek fiyat skalasına geri dönüştür
def get_predictions_real_scale(model, X, scaler):
    pred_scaled = model.predict(X, verbose=0)
    # inverse_transform: [0,1] → gerçek fiyat ($)
    return scaler.inverse_transform(pred_scaled)

y_test_real = scaler.inverse_transform(y_test)
y_pred_rnn  = get_predictions_real_scale(model_rnn, X_test, scaler)
y_pred_lstm = get_predictions_real_scale(model_lstm, X_test, scaler)

# Hata metrikleri
def eval_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:12s} | RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | MAPE: %{mape:.2f}")
    return rmse, mae, mape

print("=== TEST SETİ HATA METRİKLERİ ===")
print(f"{'Model':12s} | {'RMSE':12s} | {'MAE':12s} | {'MAPE'}")
print("-" * 55)
rnn_metrics  = eval_model('SimpleRNN',  y_test_real, y_pred_rnn)
lstm_metrics = eval_model('LSTM',       y_test_real, y_pred_lstm)

print("\nRMSE: Root Mean Squared Error (ortalama hata, $ cinsinden)")
print("MAE: Mean Absolute Error (mutlak ortalama hata, $ cinsinden)")
print("MAPE: Mean Absolute Percentage Error (yüzde hata oranı)")

In [ ]:
# Tahmin vs Gerçek fiyat grafiği
test_dates = df.index[split_idx + SEQ_LENGTH:]

plt.figure(figsize=(16, 7))

# Eğitim verisi (referans için)
train_real = scaler.inverse_transform(train_data)
plt.plot(df.index[:split_idx], train_real, color='#90CAF9', linewidth=1, 
         alpha=0.6, label='Eğitim Verisi')

# Gerçek test fiyatları
plt.plot(test_dates, y_test_real, color='#1565C0', linewidth=2, 
         label='Gerçek Fiyat (Test)')

# SimpleRNN tahmini
plt.plot(test_dates, y_pred_rnn, color='#FF5722', linewidth=1.5, 
         linestyle='--', alpha=0.9, label=f'SimpleRNN (RMSE: ${rnn_metrics[0]:.2f})')

# LSTM tahmini
plt.plot(test_dates, y_pred_lstm, color='#4CAF50', linewidth=1.5, 
         linestyle='-.', alpha=0.9, label=f'LSTM (RMSE: ${lstm_metrics[0]:.2f})')

# Eğitim/test sınırı
plt.axvline(x=df.index[split_idx], color='black', linestyle=':', 
            linewidth=2, label='Eğitim/Test Sınırı')

plt.title('SimpleRNN vs LSTM — Hisse Fiyat Tahmini', fontsize=14, fontweight='bold')
plt.xlabel('Tarih')
plt.ylabel('Fiyat ($)')
plt.legend(loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Son 60 günü yakından incele
n_zoom = 60

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_pred, name, color in zip(
    axes,
    [y_pred_rnn, y_pred_lstm],
    ['SimpleRNN', 'LSTM'],
    ['#FF5722', '#4CAF50']
):
    ax.plot(test_dates[-n_zoom:], y_test_real[-n_zoom:], 
            color='#1565C0', linewidth=2.5, label='Gerçek Fiyat')
    ax.plot(test_dates[-n_zoom:], y_pred[-n_zoom:], 
            color=color, linewidth=2, linestyle='--', label=f'{name} Tahmini')
    
    # Hata bandı — ne kadar yanılıyoruz?
    error = np.abs(y_test_real[-n_zoom:] - y_pred[-n_zoom:])
    ax.fill_between(test_dates[-n_zoom:], 
                    y_test_real[-n_zoom:].flatten() - error.flatten(),
                    y_test_real[-n_zoom:].flatten() + error.flatten(),
                    alpha=0.15, color=color, label='Hata Bandı')
    
    ax.set_title(f'{name} — Son {n_zoom} Gün (Yakın Çekim)')
    ax.set_xlabel('Tarih')
    ax.set_ylabel('Fiyat ($)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.setp(ax.get_xticklabels(), rotation=30)

plt.suptitle('Son 60 Gün Karşılaştırması', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Adım 5: Gelecek Tahmini — 30 Günlük Öngörü

Şimdiye kadar "test setindeki bilinen günleri" tahmin ettik.  
Şimdi gerçekten **gelecekteki** fiyatları tahmin edelim.

### Iterative Forecasting (Yinelemeli Tahmin)

```
Günlük 1: Son 60 günü kullan → Gün 61'i tahmin et
Gün 2:    Son 59 gerçek gün + Tahmin edilen Gün 61 → Gün 62'yi tahmin et
...
```

Her adımda kendi tahminini giriş olarak kullanır. Hata birikir → uzun vadeli tahmin zorlaşır.


In [ ]:
def forecast_future(model, last_sequence, n_steps, scaler):
    """
    n_steps gün ileriye tahmin yap.
    Her adımda önceki tahmin yeni giriş olarak kullanılır.
    """
    current_seq = last_sequence.copy()  # (60, 1)
    predictions = []
    
    for _ in range(n_steps):
        # Model girişi: (1, 60, 1)
        x_input = current_seq.reshape(1, SEQ_LENGTH, 1)
        next_val = model.predict(x_input, verbose=0)[0, 0]
        
        predictions.append(next_val)
        
        # Sırayı kaydır: en eski değeri at, yeni tahmini ekle
        current_seq = np.append(current_seq[1:], [[next_val]], axis=0)
    
    # Normalize edilmiş tahminleri gerçek fiyata dönüştür
    predictions_array = np.array(predictions).reshape(-1, 1)
    return scaler.inverse_transform(predictions_array)


# Son 60 günlük veriyi başlangıç noktası olarak kullan
last_60_days = prices_scaled[-SEQ_LENGTH:]

FORECAST_DAYS = 30
future_rnn  = forecast_future(model_rnn,  last_60_days, FORECAST_DAYS, scaler)
future_lstm = forecast_future(model_lstm, last_60_days, FORECAST_DAYS, scaler)

# Gelecek tarihler oluştur (son tarihin sonrasından itibaren iş günleri)
future_dates = pd.date_range(
    start=df.index[-1] + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq='B'
)

# prices şekli (N, 1) — scalar almak için [0] ile indeksle
son_fiyat = prices[-1][0]

print(f"Tahmin edilen dönem: {future_dates[0].date()} → {future_dates[-1].date()}")
print(f"Son bilinen fiyat: ${son_fiyat:.2f}")
print(f"\nSimpleRNN 30 gün tahmini:")
print(f"  Başlangıç: ${future_rnn[0][0]:.2f} | Bitiş: ${future_rnn[-1][0]:.2f}")
print(f"\nLSTM 30 gün tahmini:")
print(f"  Başlangıç: ${future_lstm[0][0]:.2f} | Bitiş: ${future_lstm[-1][0]:.2f}")

In [ ]:
# Gelecek tahmini görselleştir
n_history = 120   # Son 120 günü göster (bağlam için)

plt.figure(figsize=(16, 7))

# Geçmiş fiyatlar (bağlam)
plt.plot(df.index[-n_history:], prices[-n_history:],
         color='#1565C0', linewidth=2, label='Tarihsel Fiyat')

# SimpleRNN gelecek tahmini — son gerçek fiyatla birleştir
plt.plot([df.index[-1]] + list(future_dates),
         [son_fiyat] + list(future_rnn.flatten()),
         color='#FF5722', linewidth=2.5, linestyle='--',
         marker='o', markersize=4, label='SimpleRNN Tahmini')

# LSTM gelecek tahmini
plt.plot([df.index[-1]] + list(future_dates),
         [son_fiyat] + list(future_lstm.flatten()),
         color='#4CAF50', linewidth=2.5, linestyle='--',
         marker='s', markersize=4, label='LSTM Tahmini')

# Tahmin başlangıç çizgisi
plt.axvline(x=df.index[-1], color='gray', linestyle=':', linewidth=2,
            label='Bugün (Tahmin Başlangıcı)')
plt.axvspan(future_dates[0], future_dates[-1], alpha=0.05, color='yellow',
            label='Tahmin Dönemi')

plt.title('30 Günlük Hisse Fiyat Tahmini', fontsize=14, fontweight='bold')
plt.xlabel('Tarih')
plt.ylabel('Fiyat ($)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Özet tablo
print("\n=== 30 GÜNLÜK TAHMİN ÖZETİ ===")
print(f"Başlangıç fiyatı: ${son_fiyat:.2f}")
print(f"{'Model':<15} {'30.Gün Tahmini':>18} {'Değişim':>12}")
print("-" * 45)
for name, pred in [('SimpleRNN', future_rnn), ('LSTM', future_lstm)]:
    end_price = pred[-1][0]
    change = (end_price - son_fiyat) / son_fiyat * 100
    direction = "▲" if change > 0 else "▼"
    print(f"{name:<15} ${end_price:>15.2f} {direction} %{abs(change):>8.2f}")

## Sonuç

Bu notebook'ta gerçek bir fintech senaryosu üzerinde çalıştık:

1. **Gerçekçi hisse verisi** simüle ettik (trend + mevsimsellik + rastgele yürüyüş + kriz)
2. **MinMaxScaler** ve **kronolojik bölme** ile veriyi hazırladık
3. **SimpleRNN vs LSTM** karşılaştırması yaptık
4. **Iterative forecasting** ile 30 günlük öngörü ürettik
5. **RMSE/MAE/MAPE** metrikleriyle modelleri değerlendirdik

### SimpleRNN vs LSTM — Hangisi Kazandı?

LSTM genellikle hisse tahmini gibi **uzun bağımlılıklı** problemlerde SimpleRNN'i geçer.  
Bunun nedeni: LSTM'in forget/input/output gate mekanizması uzak geçmişi daha iyi korur.

### Gerçek Hayatta Ekstra Özellikler

Sadece kapanış fiyatıyla sınırlı kalmak gerekmez. Giriş özelliklerine şunlar eklenebilir:

- **Teknik göstergeler**: RSI, MACD, Bollinger Bands
- **Hacim verisi**: İşlem hacmi trend sinyali verir
- **Haber duyarlılığı**: NLP ile haber metinleri skorlanır
- **Piyasa endeksleri**: BIST100, S&P500 korelasyonu

### Önemli Uyarı

Finans modellerinde yüksek test doğruluğu gerçek getiriyi garanti etmez.  
**Backtest** (geriye dönük test) ve **transaction cost** (işlem maliyeti) de hesaba katılmalıdır.
